In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

train_df = pd.read_csv("../data/challenge1_train.csv")
test_df = pd.read_csv("../data/challenge1_test.csv")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7398, 10)
Test shape: (2467, 9)


In [2]:
X = train_df.drop(columns=["price", "ID"])
y = train_df["price"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("X_train:", X_train.shape, "| X_val:", X_val.shape)

X_train: (5918, 8) | X_val: (1480, 8)


### Model-specific Preprocessing

In [3]:
numeric_features = ["bathrooms", "bedrooms", "square_feet", "latitude", "longitude"]
categorical_features = ["category", "has_photo", "state"]

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### Baseline Random Forest

In [5]:
rf_base = Pipeline([
    ("prep", rf_preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

t0 = time.time()
rf_base.fit(X_train, y_train)
fit_time = time.time() - t0

cv_scores = -cross_val_score(rf_base, X_train, y_train, cv=cv,
                             scoring="neg_root_mean_squared_error", n_jobs=-1)

baseline = {
    "Train RMSE": rmse(y_train, rf_base.predict(X_train)),
    "CV RMSE (5-fold)": cv_scores.mean(),
    "Validation RMSE": rmse(y_val, rf_base.predict(X_val)),
}
print(f"Fit time: {fit_time:.1f}s")
for k, v in baseline.items():
    print(f"{k:18s}: {v:,.2f}")
print(f"CV RMSE std across folds: {cv_scores.std():,.2f}")

Fit time: 0.6s
Train RMSE        : 159.97
CV RMSE (5-fold)  : 424.34
Validation RMSE   : 456.55
CV RMSE std across folds: 19.97


### Grid Search

In [6]:
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [20, None],
    "model__min_samples_leaf": [1, 3, 5],
    "model__max_features": [0.33, 0.6, 1.0],
}

grid = GridSearchCV(
    estimator=Pipeline([
        ("prep", rf_preprocessor),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=1,
)

t0 = time.time()
grid.fit(X_train, y_train)
print(f"Grid Search time: {(time.time() - t0)/60:.1f} min")
print("Best params:", grid.best_params_)
print(f"Best CV RMSE: {-grid.best_score_:,.2f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Grid Search time: 2.1 min
Best params: {'model__max_depth': 20, 'model__max_features': 0.33, 'model__min_samples_leaf': 1, 'model__n_estimators': 400}
Best CV RMSE: 412.69


In [7]:
results = (pd.DataFrame(grid.cv_results_)
           .assign(cv_rmse=lambda d: -d["mean_test_score"],
                   train_rmse=lambda d: -d["mean_train_score"])
           [["param_model__n_estimators", "param_model__max_depth",
             "param_model__min_samples_leaf", "param_model__max_features",
             "cv_rmse", "train_rmse"]]
           .sort_values("cv_rmse"))
results.columns = ["n_estimators", "max_depth", "min_samples_leaf", "max_features", "cv_rmse", "train_rmse"]
results.head(10).round(2)

,n_estimators,max_depth,min_samples_leaf,max_features,cv_rmse,train_rmse
1,400,20,1,0.33,412.69,181.23
0,200,20,1,0.33,413.39,182.60
27,400,None,3,0.60,414.80,285.99
19,400,None,1,0.33,415.10,155.82
25,400,None,1,0.60,415.13,155.51
9,400,20,3,0.60,415.37,288.78
7,400,20,1,0.60,415.48,171.44
24,200,None,1,0.60,415.59,156.68
26,200,None,3,0.60,415.74,286.50
6,200,20,1,0.60,415.85,172.30


### Compare (Baseline vs Tuned)

In [8]:
best_rf = grid.best_estimator_

tuned = {
    "Train RMSE": rmse(y_train, best_rf.predict(X_train)),
    "CV RMSE": -grid.best_score_,
    "Validation RMSE": rmse(y_val, best_rf.predict(X_val)),
}

summary = pd.DataFrame({
    "Train RMSE": [baseline["Train RMSE"], tuned["Train RMSE"]],
    "CV RMSE": [baseline["CV RMSE (5-fold)"], tuned["CV RMSE"]],
    "Validation RMSE": [baseline["Validation RMSE"], tuned["Validation RMSE"]],
}, index=["RF baseline (default)", "RF tuned (grid search)"]).round(2)
summary["Gap (Val - Train)"] = (summary["Validation RMSE"] - summary["Train RMSE"]).round(2)
summary

,Train RMSE,CV RMSE,Validation RMSE,Gap (Val - Train)
RF baseline (default),159.97,424.34,456.55,296.58
RF tuned (grid search),181.80,412.69,444.87,263.07


### Feature Engineering

In [9]:
def add_features(df):
    df = df.copy()
    df["sqft_per_bedroom"] = df["square_feet"] / (df["bedrooms"] + 1)   # +1 avoids dividing by 0 for studios
    df["sqft_per_bathroom"] = df["square_feet"] / df["bathrooms"].clip(lower=0.5)
    df["total_rooms"] = df["bedrooms"] + df["bathrooms"]
    df["lat_lon_interaction"] = df["latitude"] * df["longitude"]
    return df

eng_numeric = numeric_features + ["sqft_per_bedroom", "sqft_per_bathroom", "total_rooms", "lat_lon_interaction"]

eng_preprocessor = ColumnTransformer([
    ("num", "passthrough", eng_numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

best_params = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}

rf_fe = Pipeline([
    ("prep", eng_preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params)),
])

X_train_fe, X_val_fe = add_features(X_train), add_features(X_val)

cv_scores_fe = -cross_val_score(rf_fe, X_train_fe, y_train, cv=cv,
                                scoring="neg_root_mean_squared_error", n_jobs=-1)
rf_fe.fit(X_train_fe, y_train)

fe = {
    "Train RMSE": rmse(y_train, rf_fe.predict(X_train_fe)),
    "CV RMSE": cv_scores_fe.mean(),
    "Validation RMSE": rmse(y_val, rf_fe.predict(X_val_fe)),
}
print({k: round(float(v), 2) for k, v in fe.items()})

{'Train RMSE': 173.78, 'CV RMSE': 417.11, 'Validation RMSE': 450.87}


### Final Comparison

In [10]:
final_summary = pd.concat([
    summary.drop(columns="Gap (Val - Train)"),
    pd.DataFrame([fe], index=["RF tuned + engineered features"]),
]).round(2)

final_summary["Gap (Val - Train)"] = (final_summary["Validation RMSE"] - final_summary["Train RMSE"]).round(2)
final_summary["CV change vs prev. row"] = final_summary["CV RMSE"].diff().round(2)
final_summary["Val change vs prev. row"] = final_summary["Validation RMSE"].diff().round(2)

print("Selected by lowest CV RMSE:", final_summary["CV RMSE"].idxmin())
final_summary

Selected by lowest CV RMSE: RF tuned (grid search)


,Train RMSE,CV RMSE,Validation RMSE,Gap (Val - Train),CV change vs prev. row,Val change vs prev. row
RF baseline (default),159.97,424.34,456.55,296.58,NaN,NaN
RF tuned (grid search),181.80,412.69,444.87,263.07,-11.65,-11.68
RF tuned + engineered features,173.78,417.11,450.87,277.09,4.42,6.00
